# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/malakanwarr/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [5]:
import pandas as pd

# 1. Load the fast, pre-filtered dataset from ML-04
# (Updated to the file DuckDB successfully saved for us)

df = pd.read_csv('master_dataset_ready.csv')

# 2. Isolate your chosen features (plus the ID for context)
chosen_columns = [
    'content_hash_id',
    'word_count',
    'search_volume',
    'main_intent',
    'gsc_clicks',
    'ga4_engaged_sessions'
]
model_df = df[chosen_columns].copy()

# 3. FILLS: Handle missing values
# If traffic or search volume is blank, it means there was 0 traffic.
model_df['gsc_clicks'] = model_df['gsc_clicks'].fillna(0)
model_df['ga4_engaged_sessions'] = model_df['ga4_engaged_sessions'].fillna(0)
model_df['search_volume'] = model_df['search_volume'].fillna(0)
model_df['word_count'] = model_df['word_count'].fillna(0)

# If intent is blank, give it a placeholder word before we encode it
model_df['main_intent'] = model_df['main_intent'].fillna('unknown')

# 4. CATEGORICAL HANDLING: Turn words into numbers
# This takes a column containing words like "transactional" or "informational"
# and splits it into new columns with 1s and 0s (True/False) so the model can read it.
model_df = pd.get_dummies(model_df, columns=['main_intent'], dtype=int)

# Show the cleaned, math-ready dataset!
model_df.head()

/tmp/ipykernel_2120/3129993703.py:6: DtypeWarning: Columns (24,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('master_dataset_ready.csv')


,content_hash_id,word_count,search_volume,gsc_clicks,ga4_engaged_sessions,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,main_intent_unknown
0,content_004e9c4c32e88631,3935.0,10.0,0.0,0.0,0,1,0,0,0
1,content_0236ef736698e17c,4335.0,20.0,0.0,0.0,0,0,0,1,0
2,content_025f6cfd3c298870,3719.0,10.0,0.0,0.0,1,0,0,0,0
3,content_0263d5f9b7a2ecd4,3246.0,0.0,0.0,0.0,0,1,0,0,0
4,content_02752c6c1c60161f,3641.0,20.0,0.0,0.0,1,0,0,0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

* word_count:
**Meaning:** The total number of words on the web page, representing the content's length and depth.
**Missing Values:** Filled with 0, assuming a blank value indicates the page has no indexable text or parsing failed.
**Available-when:** Exists before the prediction moment because it is a static, physical property of the page as it existed at the end of March.

* search_volume:
**Meaning:** The estimated monthly search demand for the page's primary keyword.
**Missing Values:**Filled with 0, assuming a lack of data means the keyword has negligible or zero search volume.
**Available-when:** Exists before the prediction moment as it is an established metric based on historical keyword research, not a future outcome.

* main_intent:
**Meaning:** The primary goal of the user searching for this page (e.g., informational, transactional).
**Categorical & Missing:** Missing values were filled with the placeholder string 'unknown'. The entire column was then one-hot encoded (turned into separate binary columns with 1s and 0s) so the math model can
 **Available-when:** Exists before the prediction moment because the content's purpose is defined the moment it is published.

* gsc_clicks:
**Meaning:**  The total number of clicks the page received from Google Search results.
**Missing Values:**Filled with 0, because if Google Search Console does not log a click, it means zero clicks occurred.
**Available-when:** Exists before the prediction moment because we strictly limit the sum to clicks that occurred between March 1 and March 31.

* ga4_engaged_sessions
**Meaning:** The number of user sessions where a visitor actively interacted with the page (e.g., stayed longer than 10 seconds or scrolled).
**Missing Values:** Filled with 0, as a lack of tracking data implies no recorded engagement.
**Available-when:**  Exists before the prediction moment because we only aggregate engagement that was finalized and recorded by the end of March.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [7]:
# 3. The Leakage Hunt (Security Scanner)
print("--- Initiating Leakage Hunt ---")

# TEST 1: Product Flags Attack
# System flags usually start with 'is_' (like is_published). We check if any sneaked in.
product_flags = [col for col in model_df.columns if 'is_' in col]
print(f"1. Product Flags detected: {len(product_flags)} {product_flags}")

# TEST 2: Future Windows Attack
# We check if any date/time columns survived that might secretly leak April's timeline.
# (This should be empty, as our features should only be static integers/floats).
time_columns = [col for col in model_df.columns if 'date' in col or 'time' in col]
print(f"2. Time/Future Window columns detected: {len(time_columns)} {time_columns}")

# TEST 3: Label-Derived Attack (Perfect Duplicates)
# Since we don't have a final label yet (unsupervised), the biggest cheat is if
# one feature is a 100% mathematical duplicate of another, giving it unfair weight.
duplicate_columns = model_df.columns[model_df.T.duplicated()].tolist()
print(f"3. Duplicate/Label-derived columns detected: {len(duplicate_columns)} {duplicate_columns}")

print("\n--- Final Result ---")
if len(product_flags) == 0 and len(time_columns) == 0 and len(duplicate_columns) == 0:
    print("✅ LEAKAGE TEST PASSED: No product flags, future windows, or cheating duplicates found in the final features.")
else:
    print("🚨 WARNING: Leakage detected. Clean your features.")

--- Initiating Leakage Hunt ---
1. Product Flags detected: 0 []
2. Time/Future Window columns detected: 0 []
3. Duplicate/Label-derived columns detected: 0 []

--- Final Result ---
✅ LEAKAGE TEST PASSED: No product flags, future windows, or cheating duplicates found in the final features.


"Note: The true defense against Future Window leakage was executed during the data pipeline (ML-04), where a strict SQL boundary (report_date <= '2026-03-31') was enforced before aggregation. This guarantees our gsc_clicks and ga4_sessions features contain strictly historical, pre-prediction data."

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

**Excluded:**
* is_deleted: Excluded because it was used as a pre-filter; only active pages (False) were kept, meaning the column has zero mathematical variance for the model to learn from.

* is_published: Excluded because it was used as a pre-filter; only live pages (True) were kept, leaving it with zero variance.

* sessions_paid: Excluded because this specific SEO clustering model strictly evaluates organic content performance, making paid ad traffic irrelevant noise.

* Any April 2026 Metrics (Future Windows): Excluded intentionally to prevent time-window leakage; the model must group pages based strictly on what was known by March 31.

* url_hash_id & content_hash_id: Excluded from the mathematical feature vector (kept only as context) because randomly generated ID strings carry no actual performance or structural signal.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.